# 🚀 Notebook do Professor (Demo) — Aula 04: Context Engineering De prompt para contexto

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 04/14 — Módulo 1: LangChain Foundations · 🏁 Entrega CKP01**  
**⏱️ 1h40min**  
**🧠 Context rot · XML tagging · Meta prompting**  
**🏁 CKP01 entrega**  

---

## 🎯 Objetivo da aula

Entender que o contexto é um recurso finito e caro — e aprender a preenchê-lo intencionalmente: certas informações no lugar certo, na hora certa, com o mínimo de tokens necessário. Aplicar isso ao CKP01.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções dos exercícios para executar em sala.

---

# 🔬 Código da aula — slide a slide

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

### Slide 10 — XML tagging — estruturar o contexto em seções

```
Você é um assistente de culinária
brasileira. Responda sempre em
português. Não fale sobre outros
assuntos. Se o usuário perguntar
sobre outro tema, redirecione. Seja
amigável e use emojis. Mencione
ingredientes locais. Limite respostas
a 3 parágrafos. Não use markdown.
Cite fontes quando possível...

# 8 instruções misturadas num bloco
# O modelo tende a priorizar as primeiras
# e "esquecer" as do meio e do fim
```

### Slide 10 — XML tagging — estruturar o contexto em seções

```
<persona>
  Assistente de culinária brasileira.
  Tom amigável com emojis moderados.
</persona>

<restricoes>
  - Somente tópicos de culinária
  - Redirecionar outros assuntos
  - Máximo 3 parágrafos por resposta
</restricoes>

<formato>
  Sem markdown. Sem headers.
  Ingredientes em listas simples.
</formato>

# Cada seção é processada como unidade
# Mais fácil de atualizar e depurar
```

### Slide 11 — Meta prompting — o modelo que melhora seus próprios prompts

In [ ]:
PROMPT_OTIMIZADOR = """
<tarefa>
Você é um especialista em context engineering.
Analise o system prompt abaixo e reescreva-o aplicando:
- XML tagging para separar seções
- Instruções críticas no início e no final
- Remoção de redundâncias
- Clareza nas restrições
</tarefa>

<prompt_original>
{prompt_original}
</prompt_original>

<formato_saida>
Retorne o prompt otimizado entre tags <prompt_otimizado>...</prompt_otimizado>
Depois explique em 3 bullet points as principais mudanças feitas.
</formato_saida>
"""

# Chain de meta prompting
chain_otimizador = ChatPromptTemplate.from_template(PROMPT_OTIMIZADOR) | llm | StrOutputParser()

# Usar: mandar o system prompt atual para ser melhorado
resultado = chain_otimizador.invoke({
    "prompt_original": MEU_SYSTEM_PROMPT_ATUAL
})
print(resultado)  # → versão melhorada + explicação das mudanças

### Slide 12 — Medir tokens — quanto o contexto está custando

In [ ]:
!pip install tiktoken -q

import tiktoken

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    """Conta tokens usando o tokenizador do modelo especificado."""
    enc = tiktoken.encoding_for_model(modelo)
    return len(enc.encode(texto))

# Comparar system prompt antes e depois de context engineering
ANTES = """Você é um assistente especializado em culinária brasileira.
Responda sempre em português do Brasil com um tom amigável e use emojis
moderadamente. Não fale sobre assuntos não relacionados a culinária.
Se o usuário perguntar sobre outro assunto, redirecione gentilmente.
Limite suas respostas a no máximo 3 parágrafos. Não use markdown.
Mencione ingredientes e técnicas tradicionais brasileiras sempre que
possível. Cite a região de origem dos pratos quando souber..."""

DEPOIS = """<persona>Chef assistente de culinária brasileira. Tom amigável, emojis moderados.</persona>
<restricoes>Somente culinária. Redirecionar outros temas. Máximo 3 parágrafos. Sem markdown.</restricoes>
<contexto>Priorizar ingredientes e técnicas regionais brasileiras.</contexto>"""

print(f"Antes:  {contar_tokens(ANTES)} tokens")   # → ~90 tokens
print(f"Depois: {contar_tokens(DEPOIS)} tokens")  # → ~50 tokens
print(f"Redução: {(1 - contar_tokens(DEPOIS)/contar_tokens(ANTES))*100:.0f}%")

### Slide 21 — Python novo desta aula

In [ ]:
# 1. tiktoken — contar tokens antes de enviar ao modelo
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4")
tokens = enc.encode("Meu texto aqui")  # lista de IDs
len(tokens)                              # número de tokens

# 2. Strings multilinha com XML (melhor legibilidade)
SYSTEM = """<persona>
Chef de culinária. Tom amigável.
</persona>
<restricoes>
Somente culinária. Máximo 3 parágrafos.
</restricoes>"""

# 3. f-string com cálculo percentual inline
reducao = (1 - depois / antes) * 100
print(f"Redução: {reducao:.1f}%")  # :1f → 1 casa decimal

# 4. from_template() — para prompts sem roles
prompt = ChatPromptTemplate.from_template("""
Reescreva este prompt com XML tagging:
{prompt_original}
""")  # sem roles — tratado como HumanMessage

# 5. Função com type annotation de retorno (-> int)
def contar_tokens(texto: str) -> int:
    """Documenta: recebe str, retorna int."""
    return len(enc.encode(texto))

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Context rot no código: regra no meio vs na borda

**Para o professor:** a solução roda a pergunta mista duas vezes — a regra crítica no meio do bloco (a posição que o modelo tende a esquecer, achado do "Lost in the Middle") e a mesma regra na borda final, dentro da tag `<critico>`. Compare as duas respostas na tela e feche com a regra prática do CKP01: instruções críticas no início e no fim do contexto.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain langchain-ollama langchain-core tiktoken -q

import tiktoken, re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

REGRACRITICA = "Se a pergunta sair do domínio, redirecione na primeira frase."

# Versão MISTURADA: a regra crítica no MEIO de 8 instruções
PROMPT_MISTURADO = f"""Você é um assistente de culinária. Responda sempre em português.
{REGRACRITICA}
Seja amigável e use emojis. Mencione ingredientes locais. Limite respostas a
3 parágrafos. Não use markdown. Cite fontes quando possível."""

# Versão XML: a regra crítica na ÚLTIMA borda, dentro de uma tag
PROMPT_XML = f"""<persona>Chef de culinária brasileira.</persona>
<formato>Sem markdown. Máximo 3 parágrafos.</formato>
<critico>{REGRACRITICA}</critico>"""

pergunta_teste = "Me explique o que é Docker. E aproveita: qual é a capital do Brasil?"

for nome, system in [("MISTURADA (regra no meio)", PROMPT_MISTURADO),
                     ("XML (regra na borda)", PROMPT_XML)]:
    chain = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human",  "{pergunta}"),
    ]) | llm | StrOutputParser()
    print(f"=== {nome} ===")
    print(chain.invoke({"pergunta": pergunta_teste}), "\n")
# Observação típica: a versão XML adere melhor ao redirecionamento — a
# posição da instrução no contexto muda a aderência (context rot).


### Exercício 2 — XML tagging + tiktoken: comprima sem perder instruções

**Para o professor:** a solução demonstra a compactação com um system prompt concreto de exemplo (domínio culinária) — o par ANTES/DEPOIS com `contar_tokens()` medindo a redução em percentual (tipicamente 30–50%). A regra do corte: saem polidez, repetições e conectivos; persona, restrições e formato ficam intactos. Lembre: o system prompt entra em todos os turnos do chatbot do CKP01, então cada token economizado se multiplica por turno.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain langchain-ollama langchain-core tiktoken -q

import tiktoken, re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

ANTES = """Você é um assistente especializado em culinária brasileira.
Responda sempre em português do Brasil com um tom amigável e use emojis
moderadamente. Não fale sobre assuntos não relacionados a culinária.
Se o usuário perguntar sobre outro assunto, redirecione gentilmente.
Limite suas respostas a no máximo 3 parágrafos. Não use markdown.
Mencione ingredientes e técnicas tradicionais brasileiras sempre que possível."""

DEPOIS = """<persona>Chef de culinária brasileira. Tom amigável, emojis moderados.</persona>
<restricoes>Somente culinária. Redirecionar outros temas. Máximo 3 parágrafos.</restricoes>
<formato>Sem markdown. Ingredientes em listas simples.</formato>
<contexto>Priorizar ingredientes e técnicas regionais brasileiras.</contexto>"""

tok_antes, tok_depois = contar_tokens(ANTES), contar_tokens(DEPOIS)
print(f"Antes:  {tok_antes} tokens")
print(f"Depois: {tok_depois} tokens")
print(f"Redução: {(1 - tok_depois / tok_antes) * 100:.0f}%")
# Cada seção entre tags é processada como unidade — mais fácil de
# atualizar e depurar. Checklist do corte: persona, restrições e formato
# intactos; só a polidez e as repetições saem.


### Exercício 3 — Teste A/B: aderência antes e depois do contexto

**Para o professor:** duas chains com o mesmo `llm` e o mesmo placeholder `{pergunta}` — a única variável entre elas é o system prompt, e a mesma pergunta mista (pedido do domínio + tema de fora) garante a validade do A/B. Na observação típica, a versão XML redireciona o tema de fora e mantém o limite de parágrafos; a versão bagunçada tende a vazar do domínio. Feche com o veredito: tokens antes/depois + qualidade equivalente = context engineering venceu.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain langchain-ollama langchain-core tiktoken -q

import tiktoken, re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

ANTES = """Você é um assistente especializado em culinária brasileira.
Responda sempre em português do Brasil com um tom amigável e use emojis
moderadamente. Não fale sobre assuntos não relacionados a culinária.
Se o usuário perguntar sobre outro assunto, redirecione gentilmente.
Limite suas respostas a no máximo 3 parágrafos. Não use markdown.
Mencione ingredientes e técnicas tradicionais brasileiras sempre que possível."""

DEPOIS = """<persona>Chef de culinária brasileira. Tom amigável, emojis moderados.</persona>
<restricoes>Somente culinária. Redirecionar outros temas. Máximo 3 parágrafos.</restricoes>
<formato>Sem markdown. Ingredientes em listas simples.</formato>
<contexto>Priorizar ingredientes e técnicas regionais brasileiras.</contexto>"""

chain_antes = ChatPromptTemplate.from_messages([
    ("system", ANTES),        # prompt bagunçado (~90 tokens)
    ("human",  "{pergunta}"),
]) | llm | StrOutputParser()

chain_depois = ChatPromptTemplate.from_messages([
    ("system", DEPOIS),       # XML tagging (~50 tokens)
    ("human",  "{pergunta}"),
]) | llm | StrOutputParser()

# Pergunta mista: 1 do domínio + 1 de fora — testa aderência e redirecionamento
pergunta_teste = "Como faço um strogonoff? E qual é a capital do Brasil?"

print("=== ANTES ===\n",  chain_antes.invoke({"pergunta": pergunta_teste}))
print("=== DEPOIS ===\n", chain_depois.invoke({"pergunta": pergunta_teste}))
# Observação típica: a versão com tags respeita melhor as restrições
# (redireciona o tema de fora e mantém o limite de parágrafos),
# com menos tokens de instrução por chamada.


### Exercício 4 — Meta prompting: o modelo otimiza o próprio prompt

**Para o professor:** a chain otimizadora é LCEL comum (`from_template` → `ChatOllama(model="gpt-oss:120b")` → `StrOutputParser()`) com o meta-prompt em XML — `<tarefa>` isola a instrução, `<prompt_original>` isola o dado e `<formato_saida>` fixa a entrega. A extração entre tags com `re.search(..., re.S)` e a medição com `contar_tokens()` mostram o ganho em números (antes × depois × redução %). Feche com a validação humana: conferir se persona, restrições e formato sobreviveram — o juiz final é o teste A/B do Exercício 3.


In [ ]:
# Setup da aula — imports e conexão com o Ollama Cloud (Colab Secrets)
!pip install langchain langchain-ollama langchain-core tiktoken -q

import tiktoken, re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

PROMPT_OTIMIZADOR = """<tarefa>
Você é um especialista em context engineering. Reescreva o system prompt
aplicando: XML tagging para separar seções, instruções críticas no início
e no final, remoção de redundâncias e clareza nas restrições.
</tarefa>

<prompt_original>
{prompt_original}
</prompt_original>

<formato_saida>
Retorne o prompt otimizado entre as tags <prompt_otimizado>...</prompt_otimizado>.
Depois explique em 3 bullet points as principais mudanças.
</formato_saida>"""

chain_otimizadora = ChatPromptTemplate.from_template(PROMPT_OTIMIZADOR) | llm | StrOutputParser()

MEU_PROMPT = """Você é um assistente especializado em culinária brasileira.
Responda sempre em português com tom amigável e emojis moderados. Não fale
sobre assuntos não relacionados. Se perguntarem outro assunto, redirecione.
Máximo 3 parágrafos. Sem markdown. Mencione ingredientes brasileiros."""

resultado = chain_otimizadora.invoke({"prompt_original": MEU_PROMPT})
print(resultado)

# Extração e medição do ganho
entre_tags = re.search(r"<prompt_otimizado>(.*?)</prompt_otimizado>", resultado, re.S)
if entre_tags:
    novo = entre_tags.group(1).strip()
    antes  = contar_tokens(MEU_PROMPT)
    depois = contar_tokens(novo)
    print(f"\nTokens antes: {antes} | depois: {depois} | redução: {(1 - depois/antes)*100:.0f}%")
# Validação humana obrigatória: conferir se persona, restrições e formato
# sobreviveram — a validação final é o teste A/B do Exercício 3.


## 📚 Referências da aula

- Blog Anthropic Engineering — "Effective Context Engineering for AI Agents" (setembro, 2025). A fonte primária do termo e das técnicas desta aula. anthropic.com/engineering/building-effective-agents
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. Base empírica do context rot. arxiv.org/abs/2307.03172
- Docs Anthropic — Prompt Library e guia de XML tagging. docs.anthropic.com/pt/docs/build-with-claude/prompt-engineering/use-xml-tags
- Docs tiktoken — Biblioteca de tokenização da OpenAI. Funciona como aproximação para qualquer modelo baseado em BPE. github.com/openai/tiktoken
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 22 — A importância do contexto na inferência linguística: a base teórica de por que o contexto é informação.

---

**→ Próxima Aula — Aula 05 · 31/08** — Embeddings e busca semântica com ChromaDB
  
Transformar texto em vetores e buscar por similaridade. A fundação do RAG começa aqui.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*